# Read database, split into chunks, vectorize and add to vectorDB

In [7]:
from pathlib import Path

# Load all documents from database at ./knowledge-base
documents = []

for folder in Path("knowledge-base").iterdir(): # Finds folders at 1st level only. Non recursive.
    doc_type = folder.name
    for file in folder.rglob("*.md"): # Find all files and directories recursively matching pattern *.md
        with open(file, "r", encoding="utf-8") as f:
            documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()}) # as_posix() gives forward slashes in path even for windows

print(f"Loaded {len(documents)} documents\nsample {documents[0]}")

Loaded 76 documents
sample {'type': 'company', 'source': 'knowledge-base/company/about.md', 'text': "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a po

In [8]:
from tqdm import tqdm
from pydantic import BaseModel, Field
from litellm import completion
import instructor

# Use LLM to break the documents into chunks

chunks = [] # shall hold list of pydantic class Result

class Result(BaseModel):
    page_content: str
    metadata: dict

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document): # return as pydantic object of class Result
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
                    metadata={"source": document["source"], "type": document["type"]})

class Chunks(BaseModel):
    chunks: list[Chunk]

AVERAGE_CHUNK_SIZE = 500
def create_user_prompt(document):
    avg_n_chunks = len(document["text"]) // AVERAGE_CHUNK_SIZE + 1

    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {avg_n_chunks} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

for document in tqdm(documents): # tqdm adds a nice progress bar for loops
    message = [ # LLM message per document to break it into chunks
        {"role": "user", "content": create_user_prompt(document)}
    ]

    litellm = instructor.from_litellm(completion) # Use instructor to ensure model output is strict json.
    response = litellm.chat.completions.create(model="ollama/gpt-oss:120b-cloud", messages=message, response_model=Chunks) # returns class Chunk
    doc_as_chunk = response.chunks
    chunks.extend([chunk.as_result(document) for chunk in doc_as_chunk])

print(f"Created {len(chunks)} chunks\n sample {chunks[0]}")

100%|██████████| 76/76 [40:17<00:00, 31.81s/it]

Created 606 chunks
 sample page_content='Founding and Early Growth\n\nInsurellm was founded in 2015 by Avery Lancaster as an insurance‑tech startup. Its first product, Markellm, linked consumers to insurers. In the next five years the firm added Carllm, Homellm and Rellm, and by 2020 it peaked at 200 employees across 12 U.S. offices.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growt

In [9]:
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
import os

# Convert chunks to vectors and add them to vectorDB

vector_db_name = "vector_db"
vector_db = PersistentClient(vector_db_name) # init chroma vectorDB

if os.path.exists(vector_db_name): # Reset vectorDB if it already exists. Delete all collections
    for c in vector_db.list_collections():
        vector_db.delete_collection(c.name)
        print(f"deleted vectorDB collection {c.name}")

collection = vector_db.get_or_create_collection("vectordb") # create a new collection in vectorDB

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2") # Embedding model
embeddings = model.encode([chunk.page_content for chunk in chunks]) # Convert all chunks to vectors. Chunk is class Result. Only Result.page_content is to be converted to vector.

collection.add(ids=[str(i) for i in range(len(chunks))],
               embeddings=embeddings,
               documents=[chunk.page_content for chunk in chunks],
               metadatas=[chunk.metadata for chunk in chunks])

print(f"VectorDB created with {collection.count()} documents")

deleted vectorDB collection vectordb


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorDB created with 606 documents
